# Phase 1: Automotive Document Data Ingestion and Processing

Welcome! This notebook demonstrates the foundation of our Question Answering system.
In Phase 1, our goal is to convert messy raw documents (PDFs, Images, Excel sheets) into clean, bite-sized facts that the AI can rapidly search through.

### What to expect:
1. **Ingestion**: Loading files (you can upload multiple files at once).
2. **Cleaning & Metadata**: Stripping garbage text and adding structured labels.
3. **Chunking**: Cutting documents into overlapping chunks.
4. **Embedding**: Using a SentenceTransformer model to convert text into math vectors.
5. **Indexing**: Building our offline, CPU-friendly FAISS database.

In [ ]:
!pip install -r ../requirements.txt
import sys
import os
sys.path.append(os.path.abspath('..'))

In [ ]:
from src.processing import process_document
from src.chunking import recursive_chunking
from src.embeddings import generate_embeddings
from src.vector_store import VectorStore
import shutil

print("Libraries successfully loaded. Ready to begin Phase 1 pipeline!")

## 1. Upload and Process Documents
Run this block to upload files from your local computer into Colab. You can upload multiple files simultaneously (e.g., PDF, DOCX, TXT, CSV, XLSX, JPG, PNG). The system will automatically route and extract them.

In [ ]:
try:
    from google.colab import files
    print("Please upload your automotive documents:")
    uploaded = files.upload()
    
    # Create a temporary directory to store uploaded files
    os.makedirs('uploaded_docs', exist_ok=True)
    
    file_paths = []
    for filename, data in uploaded.items():
        path = os.path.join('uploaded_docs', filename)
        with open(path, 'wb') as f:
            f.write(data)
        file_paths.append(path)
        print(f"Saved {filename}")
except ImportError:
    print("Not running in Google Colab. Falling back to test assets...")
    file_paths = ['../test_assets/sample.pdf', '../test_assets/sample.docx']

## 2 & 3. Batch Extraction, Cleaning, and Chunking
The system will iterate through all uploaded files, extract text, and chunk them appropriately.

In [ ]:
all_chunks = []
all_metadatas = []

for path in file_paths:
    print(f"\nProcessing: {path}")
    text, metadata = process_document(path)
    
    chunks = recursive_chunking(text)
    metadatas = [metadata.copy() for _ in chunks]
    
    all_chunks.extend(chunks)
    all_metadatas.extend(metadatas)
    
    print(f"-> Extracted {len(chunks)} chunks from {os.path.basename(path)}.")

print(f"\nTotal chunks extracted across all files: {len(all_chunks)}")

## 4 & 5. Embedding and Indexing
Finally, we run all collected chunks through our lightweight offline model (`all-MiniLM-L6-v2`) and save them into the FAISS vector database.

In [ ]:
if all_chunks:
    embeddings = generate_embeddings(all_chunks)
    
    vs = VectorStore()
    vs.build_index(embeddings, all_chunks, all_metadatas)
    vs.save_index('../faiss_index')
    
    print("\n✅ Phase 1 Successful! Database built and saved to disk.")
else:
    print("No chunks generated. Please upload valid files.")